<a href="https://colab.research.google.com/github/hawooh/Adult-Set-Decision-Tree-/blob/main/Fake%20News%20Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fake News Detection

##### Importing Lib

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import re
import string

In [4]:
from google.colab import files

In [5]:
# Upload the file
uploaded = files.upload()

Saving Fake.csv to Fake (1).csv
Saving True.csv to True (1).csv


In [6]:
import pandas as pd
# Read the CSV file (replace 'Fake.csv' with the exact filename)
df_Fake = pd.read_csv('Fake.csv')
df_Fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [7]:
# Read the CSV file (replace 'Fake.csv' with the exact filename)
df_True = pd.read_csv('True.csv')
df_True.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [9]:
df_Fake.shape, df_True.shape

((23481, 5), (21417, 5))

In [10]:
# Add a label column to each DataFrame
df_True['label'] = 1  # Label 1 for true news
df_Fake['label'] = 0  # Label 0 for fake news

# Combine the datasets
df_combined = pd.concat([df_True, df_Fake], ignore_index=True)

# Shuffle the combined dataset to mix true and fake news
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)


In [12]:
df_combined.head()

,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",0
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",0
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",1
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",0


In [13]:
print(df_combined.isnull().sum())
print(df_combined.duplicated().sum())


title      0
text       0
subject    0
date       0
label      0
dtype: int64
209


In [14]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk

# Download NLTK stopwords (if you haven't already)
nltk.download('stopwords')

# Clean the text by removing special characters, numbers, and stopwords
def Text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove special characters and numbers using regular expressions
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in text.split() if word not in stop_words])

    return text

# Apply the cleaning function to the 'text' column in the dataset
df_combined['Text'] = df_combined['text'].apply(Text)

# Define X (features) as the cleaned text and y (labels) as the label column
X = df_combined['Text']
y = df_combined['label']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the first few rows of the cleaned text and labels
df_combined[['Text', 'label']].head()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Text,label
0,donald trump white house chaos trying cover ru...,0
1,donald trump presumptive gop nominee time reme...,0
2,mike pence huge homophobe supports exgay conve...,0
3,san francisco reuters california attorney gene...,1
4,twisted reasoning comes pelosi days especially...,0


#### Step 1: Split Features and Labels

In [20]:
# Define X (features) as the cleaned text and y (labels) as the label column
X = df_combined['Text']
y = df_combined['label']


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit and transform the 'text' column to create the feature matrix
X = vectorizer.fit_transform(df_combined['Text']).toarray()

# Define the labels (y)
y = df_combined['label']


In [22]:
from sklearn.model_selection import train_test_split

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


In [23]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model
model = LogisticRegression()

# Train the model on the training data
model.fit(X_train, y_train)


LogisticRegression()

##### Make Predictions:

In [24]:
# Make predictions on the test set
y_pred = model.predict(X_test)


##### Evaluate the Model:

In [25]:
from sklearn.metrics import classification_report, confusion_matrix

# Display classification report (precision, recall, F1-score)
print(classification_report(y_test, y_pred))

# Display confusion matrix
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5877
           1       0.98      0.99      0.99      5348

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225

[[5793   84]
 [  60 5288]]


##### Step 5: Model Optimization
Use Grid Search to find the best hyperparameters for the Logistic Regression model.

python
Copy code
